# 🏥 Eleza backend on Google Colab (free T4 GPU)

This notebook runs the **Eleza backend** (FastAPI + Whisper speech-to-text) and exposes it with **ngrok**
so the Flutter web kiosk can use it.

**There is no audio file to upload.** You test by talking to the kiosk in Chrome: the browser records your
voice and sends it to this server (`POST /kiosk/{id}/transcribe`). All other AI steps run in the browser through
Firebase AI Logic (Gemini Nano on-device, Gemini in the cloud as fallback).

## Before you run anything
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Secrets** (🔑 icon in the left sidebar), each with **Notebook access ON**:

| Secret | Needed | What |
|---|---|---|
| `HF_TOKEN` | ✅ | Hugging Face read token (Whisper downloads) |
| `NGROK_AUTH_TOKEN` | ✅ | [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken) |
| `ELEVENLABS_API_KEY` | ✅ | Kiosk voice (`POST /tts`) |
| `DB_HOST` `DB_PORT` `DB_NAME` `DB_USER` `DB_PASSWORD` | ✅ | Postgres (Neon: *Connection details*; port `5432`) |
| `DEMO_PASSWORD` | ✅ first run | Password for the 3 demo accounts (min 8 characters, not one you use elsewhere) |
| `FIREBASE_SERVICE_ACCOUNT_JSON` | optional | Whole service-account JSON → server-side Remote Config for triage thresholds |

3. Set `REPO_BRANCH` in step 2, then run the cells in order (**Runtime → Run all** works too).

## 1. GPU check

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("❌ No GPU — Runtime → Change runtime type → T4 GPU → Save, then run this cell again")

## 2. Get the code
Set `REPO_BRANCH` to the branch you are testing (`main` once everything is merged).

In [ ]:
import os
from pathlib import Path

REPO_BRANCH = 'main'
REPO = Path('/content/eleza_hybrid_ai_hackathon')

if REPO.exists():
    os.chdir(REPO)
    !git fetch -q origin && git checkout -q {REPO_BRANCH} && git reset -q --hard origin/{REPO_BRANCH}
else:
    !git clone -q -b {REPO_BRANCH} https://github.com/buwituze/eleza_hybrid_ai_hackathon.git {REPO}
os.chdir(REPO / 'backend')
!git log --oneline -1

## 3. Install dependencies (~2 min)

In [ ]:
!pip install -q -r requirements.txt pyngrok
print("✅ Dependencies installed")

## 4. Load secrets

In [ ]:
import os
from google.colab import userdata

def secret(name, default=''):
    try:
        return (userdata.get(name) or default).strip()
    except Exception:
        return default

for name in ['HF_TOKEN', 'NGROK_AUTH_TOKEN', 'ELEVENLABS_API_KEY', 'DB_HOST', 'DB_PORT', 'DB_NAME',
             'DB_USER', 'DB_PASSWORD', 'DEMO_PASSWORD', 'FIREBASE_SERVICE_ACCOUNT_JSON']:
    os.environ[name] = secret(name)

os.environ['DB_PORT'] = os.environ['DB_PORT'] or '5432'
os.environ['USE_DB'] = 'true'
os.environ['DEVICE'] = 'cuda'
os.environ['MAX_TURNS'] = '10'
os.environ['SKIP_LANG_DETECTION_WHEN_HINTED'] = 'true'
if 'neon.tech' in os.environ['DB_HOST']:
    os.environ['PGSSLMODE'] = 'require'   # Neon only accepts SSL connections

def status(name, required=True):
    if os.environ.get(name):
        return '✅'
    return '❌ missing' if required else '— not set (optional)'

for name in ['HF_TOKEN', 'NGROK_AUTH_TOKEN', 'ELEVENLABS_API_KEY', 'DB_HOST', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']:
    print(f"{name:<32}{status(name)}")
print(f"{'DEMO_PASSWORD':<32}{status('DEMO_PASSWORD', False)}")
print(f"{'FIREBASE_SERVICE_ACCOUNT_JSON':<32}{status('FIREBASE_SERVICE_ACCOUNT_JSON', False)}")

## 5. Prepare the database
Creates the tables on an empty database, and the demo accounts if `DEMO_PASSWORD` is set.
Safe to run every time: existing tables and accounts are left alone.

In [ ]:
!python setup_db.py

## 6. Start the server + ngrok
Runs in the background, so the cells below stay usable. Whisper loads inside the server
(first time ~3 min: downloads ~6 GB).

In [ ]:
import os, subprocess, time, requests
from pyngrok import conf, ngrok

PORT = 8000
LOG = '/content/server.log'

# Stop anything from an earlier run (also survives a "Restart session")
ngrok.kill()
subprocess.run(['pkill', '-f', 'uvicorn main:app'])
time.sleep(3)  # let the old server release port 8000 and the GPU

server = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=open(LOG, 'w'), stderr=subprocess.STDOUT,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'},  # logs appear immediately
)
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError('Server stopped during startup — run the "Server logs" cell')
    try:
        if requests.get(f'http://localhost:{PORT}/health', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Server did not start — run the "Server logs" cell')

# Token set in memory: ngrok.set_auth_token() fails on a re-run
# ("Authtoken already set in configuration file").
conf.get_default().auth_token = os.environ['NGROK_AUTH_TOKEN']
API_URL = ngrok.connect(PORT, 'http').public_url
print("✅ Server running")
print(f"🚀 API URL: {API_URL}")
print('Put it in frontend/firebase.env.json:')
print(f'   "API_BASE_URL": "{API_URL}"')

## 7. Is it ready?
Re-run until `models_ready` is `True` (Whisper is loading).

In [ ]:
import json, requests
s = requests.get(f'http://localhost:{PORT}/startup/status', timeout=10).json()
print(f"models_ready   : {s['models_ready']}   (Whisper)")
print(f"database_ready : {s['database_ready']}")
rc = s.get('remote_config', {})
print(f"remote_config  : {rc.get('status')}")
print('   ' + json.dumps(rc.get('values', {})))
for e in s.get('startup_errors', []):
    print('❌', e)
print('\n' + s.get('message', ''))

## 8. Server logs (when something fails)

In [ ]:
!tail -n 60 /content/server.log

## 9. Stop the server

In [ ]:
import subprocess
from pyngrok import ngrok
ngrok.kill()
subprocess.run(['pkill', '-f', 'uvicorn main:app'])
print('🛑 Stopped')

## Troubleshooting
- **`database_ready: False`** → check the `DB_*` secrets. With Neon, `DB_HOST` looks like `ep-xxxx.region.aws.neon.tech`.
- **Demo accounts can't log in** → `DEMO_PASSWORD` was missing when step 5 ran: set it and run step 5 again.
- **Kiosk shows a network/CORS error** → a new run of step 6 gives a new ngrok URL: update `API_BASE_URL` and restart Flutter.
- **Colab disconnects** after ~90 min idle → keep the tab active (e.g. the *Colab Alive* extension), then re-run steps 4–7.
- **`remote_config: defaults (no Server template published yet)`** → normal until you publish the Remote Config **Server** template in the Firebase console.
- **New ngrok URL every run** → the free ngrok plan gives a new URL each time step 6 runs: update `API_BASE_URL` each time.